In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

SEED = 39
rng = np.random.default_rng(SEED)

In [2]:
# Create the movie Dataset
movies = pd.DataFrame([
    ("m01", "Moon Harbor", "A lonely astronaut discovers a quiet garden on the moon.", "science fiction|drama", [0.95, 0.15, 0.05, 0.25]),
    ("m02", "Red Planet Run", "A crew races across Mars during a dangerous rescue mission.", "science fiction|action", [0.90, 0.10, 0.80, 0.05]),
    ("m03", "The Last Signal", "A radio operator searches for survivors after a global blackout.", "science fiction|thriller", [0.75, 0.20, 0.45, 0.45]),
    ("m04", "Cedar Street", "Neighbors rebuild their community garden and become friends.", "drama|comedy", [0.05, 0.25, 0.05, 0.80]),
    ("m05", "Second Chances", "Two old friends reunite and reconsider the paths they chose.", "drama|romance", [0.05, 0.20, 0.05, 0.70]),
    ("m06", "Weekend Alibi", "A mistaken identity turns a family holiday into a comic caper.", "comedy|mystery", [0.05, 0.45, 0.35, 0.35]),
    ("m07", "The Glass Key", "A careful detective follows clues through a stormy seaside town.", "mystery|thriller", [0.10, 0.75, 0.25, 0.20]),
    ("m08", "Night Archive", "A historian uncovers a hidden story in an abandoned library.", "mystery|drama", [0.10, 0.80, 0.10, 0.35]),
    ("m09", "Wild Current", "A kayaker journeys through remote rivers to protect a forest.", "adventure|drama", [0.30, 0.15, 0.65, 0.45]),
    ("m10", "Skyline Chase", "A daring pilot races to deliver a cure across the mountains.", "action|adventure", [0.25, 0.10, 0.95, 0.15]),
    ("m11", "Little Big Kitchen", "A new chef enters a cooking contest and finds a chosen family.", "comedy|drama", [0.02, 0.10, 0.10, 0.80]),
    ("m12", "Orbiting Us", "Two engineers on a space station learn to trust one another.", "science fiction|romance", [0.85, 0.10, 0.10, 0.55]),
    ("m13", "Paper Detectives", "A pair of siblings solve neighborhood mysteries after school.", "mystery|comedy", [0.03, 0.60, 0.20, 0.45]),
    ("m14", "Deep Blue Rescue", "Divers search a shipwreck before a powerful storm arrives.", "adventure|thriller", [0.15, 0.50, 0.80, 0.10]),
    ("m15", "A Very Small Planet", "A botanist studies strange plants on a distant world.", "science fiction|adventure", [0.90, 0.25, 0.35, 0.15]),
    ("m16", "Sunday Table", "A family gathers for a meal and works through an old disagreement.", "drama|comedy", [0.02, 0.10, 0.03, 0.85])
], columns=[
    "movie_id", "title", "description", "genres", "movie_factors"
])


display(movies[["movie_id", "title", "genres"]])
print("Total Movies:", len(movies))

,movie_id,title,genres
0,m01,Moon Harbor,science fiction|drama
1,m02,Red Planet Run,science fiction|action
2,m03,The Last Signal,science fiction|thriller
3,m04,Cedar Street,drama|comedy
4,m05,Second Chances,drama|romance
5,m06,Weekend Alibi,comedy|mystery
6,m07,The Glass Key,mystery|thriller
7,m08,Night Archive,mystery|drama
8,m09,Wild Current,adventure|drama
9,m10,Skyline Chase,action|adventure


Total Movies: 16


In [3]:
# Generate users and ratings
factor_names = ["scifi", "mystery", "action", "warmth"]

movie_factors = np.array(movies["movie_factors"].to_list())

user_names = [f"user_{i:02d}" for i in range(1, 25)]
user_factors = rng.uniform(0.05, 1.0, 
                          size=(len(user_names), len(factor_names)))

user_factors[:6, 0] += 0.45
user_factors[6:12, 1] += 0.45
user_factors[12:18, 2] += 0.45
user_factors[18:, 3] += 0.45

user_factors = np.clip(user_factors, 0, 1)

rows = []

for ui, user in enumerate(user_names):
    seen = rng.choice(len(movies), size=11, replace=False)
    raw = 1.0 + 4.0 * (
        (user_factors[ui] @ movie_factors[seen].T)
        / user_factors[ui].sum()
    )
    rating_values = np.clip(np.rint(raw + rng.normal(0, 0.45, len(seen))), 1, 5).astype(int)

    for mi, rating in zip(seen, rating_values):
        rows.append((user, movies.iloc[mi]["movie_id"],
                    int(rating)))


ratings = pd.DataFrame(rows, columns=["user_id", "movie_id", "rating"])

print("Total Users:", ratings["user_id"].nunique())
print("Total Movies:", movies["movie_id"].nunique())
print("Total observed ratings:", len(ratings))

display(ratings.head(10))

Total Users: 24
Total Movies: 16
Total observed ratings: 264


,user_id,movie_id,rating
0,user_01,m08,2
1,user_01,m10,2
2,user_01,m15,3
3,user_01,m07,3
4,user_01,m04,2
5,user_01,m16,2
6,user_01,m13,2
7,user_01,m09,2
8,user_01,m14,2
9,user_01,m06,2


In [4]:
# Create the user_item ratings tabel
user_item_matrix = ratings.pivot(index="user_id", columns="movie_id", values="rating")

display(user_item_matrix)

movie_id,m01,m02,m03,m04,m05,m06,m07,m08,m09,m10,m11,m12,m13,m14,m15,m16
user_id,,,,,,,,,,,,,,,,
user_01,NaN,NaN,3.0,2.0,NaN,2.0,3.0,2.0,2.0,2.0,NaN,NaN,2.0,2.0,3.0,2.0
user_02,3.0,3.0,3.0,2.0,2.0,3.0,3.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,3.0,1.0
user_03,3.0,3.0,3.0,1.0,NaN,2.0,3.0,2.0,NaN,NaN,NaN,NaN,2.0,2.0,3.0,2.0
user_04,3.0,3.0,4.0,NaN,NaN,NaN,2.0,2.0,NaN,2.0,NaN,3.0,2.0,2.0,3.0,1.0
user_05,3.0,3.0,3.0,2.0,1.0,2.0,3.0,NaN,3.0,NaN,1.0,2.0,NaN,NaN,NaN,2.0
user_06,2.0,NaN,3.0,1.0,NaN,2.0,NaN,2.0,2.0,3.0,2.0,NaN,2.0,NaN,3.0,1.0
user_07,2.0,NaN,3.0,NaN,1.0,2.0,2.0,NaN,NaN,NaN,2.0,2.0,2.0,2.0,3.0,2.0
user_08,2.0,2.0,2.0,2.0,2.0,NaN,3.0,3.0,2.0,3.0,NaN,NaN,2.0,NaN,2.0,NaN
user_09,2.0,NaN,3.0,2.0,3.0,NaN,4.0,2.0,NaN,2.0,3.0,3.0,NaN,3.0,NaN,2.0


In [5]:
print("Unknown Ratings:", user_item_matrix.isna().sum().sum())
print("Known Ratings:", user_item_matrix.notna().sum().sum())

Unknown Ratings: 120
Known Ratings: 264


**Conclusion -->**
* A blank cell represents an unknown rating because the user has not rated that movie in the observed data. It does not mean zero stars or that the user dislikes the movie.

# **Activity 2: Content-based filtering using TF-IDF**

In [6]:
# Prepare movie text and calculate TF-IDF
movies["text"] = (movies["title"] + " " +
                 movies["genres"].str.replace("|", " ", regex=False) + " " +
                 movies["description"])

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))

item_vectors = vectorizer.fit_transform(movies["text"])
item_sim = cosine_similarity(item_vectors)

movie_pos = {movie_id: i
            for i, movie_id in enumerate(movies["movie_id"])}

print("TF-IDF matrix shape:", item_vectors.shape)

TF-IDF matrix shape: (16, 278)


In [7]:
# Create a content based recommendation function
def content_scores(user_id, history_data):
    history = history_data[history_data["user_id"] == user_id]
    liked = history[history["rating"] >= 4]

    if liked.empty:
        liked = history.nlargest(3, "rating")
    weights = liked["rating"].to_numpy(dtype=float) - 2.5

    profile = np.average(item_vectors[
        [movie_pos[m] for m in liked["movie_id"]]
    ].toarray(), axis=0, weights=weights)

    scores = item_vectors @ profile
    scores = np.asarray(scores).ravel()

    already_seen = set(history["movie_id"])

    recommendations = pd.Series(scores, index=movies["movie_id"])
    recommendations = recommendations.drop(labels=list(already_seen), errors="ignore")

    return recommendations.sort_values(ascending=False)

In [8]:
# Display the top 5 recommendations
def show_recommendations(scores, n=5):
    result = (scores.head(n).rename("score").rename_axis("movie_id").reset_index())
    result = result.merge(movies[["movie_id", "title", "genres"]], 
                         on="movie_id")

    return result[["title", "genres", "score"]]

In [9]:
# Generate recommendations for two users
sample_users = ["user_03", "user_10"]

for user in sample_users:
    print("\nUser:", user)
    print("-" * 15)
    print("\nPreviously rated movies:")
    print("-" * 25)
    history = ratings[ratings["user_id"] == user].merge(movies[["movie_id", "title"]], on="movie_id")

    display(history.sort_values("rating", ascending=False)
           [["title", "rating"]])
    
    print("\nTop-5 Content based recommendations:")
    print("-" * 37)
    display(show_recommendations(content_scores(user, ratings)))


User: user_03
---------------

Previously rated movies:
-------------------------


,title,rating
1,The Last Signal,3
6,The Glass Key,3
4,Red Planet Run,3
10,A Very Small Planet,3
8,Moon Harbor,3
2,Paper Detectives,2
0,Night Archive,2
5,Weekend Alibi,2
3,Sunday Table,2
7,Deep Blue Rescue,2



Top-5 Content based recommendations:
-------------------------------------


,title,genres,score
0,Orbiting Us,science fiction|romance,0.050742
1,Skyline Chase,action|adventure,0.026436
2,Second Chances,drama|romance,0.000000
3,Wild Current,adventure|drama,0.000000
4,Little Big Kitchen,comedy|drama,0.000000



User: user_10
---------------

Previously rated movies:
-------------------------


,title,rating
9,Weekend Alibi,3
6,The Glass Key,3
4,Skyline Chase,3
2,Moon Harbor,2
1,Paper Detectives,2
0,Cedar Street,2
3,Sunday Table,2
5,Night Archive,2
8,Orbiting Us,2
10,Little Big Kitchen,2



Top-5 Content based recommendations:
-------------------------------------


,title,genres,score
0,Red Planet Run,science fiction|action,0.026436
1,Deep Blue Rescue,adventure|thriller,0.018963
2,The Last Signal,science fiction|thriller,0.010988
3,Wild Current,adventure|drama,0.009331
4,A Very Small Planet,science fiction|adventure,0.009150


# **Activity 3: Collaborative filtering using TruncatedSVD**

In [10]:
# Split ratings into training and testing Data
train_parts = []
test_parts = []

for user, group in ratings.groupby("user_id", sort=True):
    shuffled = group.sample(frac=1, random_state=SEED + int(user[-2:]))

    test_parts.append(shuffled.head(2))
    train_parts.append(shuffled.iloc[2:])

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

user_ids = sorted(ratings["user_id"].unique())
item_ids = list(movies["movie_id"])

u_pos = {user: 1
        for i, user in enumerate(user_ids)}

i_pos = {movie: 1
        for i, movie in enumerate(item_ids)}

R = np.full((len(user_ids), len(item_ids)), np.nan)

for row in train.itertuples(index=False):
    R[u_pos[row.user_id], i_pos[row.movie_id]] = row.rating

print("Training matrix shape:", R.shape)
print("Observed training ratings:", np.isfinite(R).sum())
print("Held-Out ratings:", len(test))
print("Unknown cells:", np.isnan(R).sum())

Training matrix shape: (24, 16)
Observed training ratings: 1
Held-Out ratings: 48
Unknown cells: 383


In [11]:
# Fit the TruncatedSVD Model
train_global = train["rating"].mean()
user_mean = (train.groupby("user_id")["rating"]
            .mean()
            .reindex(user_ids)
            .fillna(train_global)
            .to_numpy())

centered = np.where(np.isfinite(R), R - user_mean[:, None], 0.0)

rank = 4

svd = TruncatedSVD(n_components=rank, random_state=SEED)
user_factors_svd = svd.fit_transform(centered)
item_factors_svd = svd.components_.T

latent = user_factors_svd @ item_factors_svd.T

pred_matrix = np.clip(latent + user_mean[:, None], 1, 5)

print("SVD components:", rank)
print("Explained Variance Ratio:",
     round(svd.explained_variance_ratio_.sum(), 4))

SVD components: 4
Explained Variance Ratio: 1.0


* Unknown ratings are represented by NaN in the original ratings matrix. The model fills their centered values with zero for this simplified SVD approach; this does not mean those unknown ratings are actual zero-star ratings.

In [12]:
# Predict held-out ratings and calculate RMSE
def predict_cf(user_id, movie_id):
    return float(pred_matrix[u_pos[user_id], i_pos[movie_id]])

cf_test_pred = np.array([predict_cf(row.user_id, row.movie_id)
                        for row in test.itertuples(index=False)])

rmse = np.sqrt(mean_squared_error(test["rating"], cf_test_pred))

print("Held-out rating RMSE:", round(rmse, 4))

Held-out rating RMSE: 0.7217


* A lower RMSE means the predicted ratings are closer to the actual held-out ratings. It does not necessarily mean the top-five recommendations are better.

In [13]:
# Rank unseen movies for collaborative filtering
def cf_scores(user_id):
    seen = set(train.loc[train["user_id"] == user_id, "movie_id"])
    scores = pd.Series(pred_matrix[u_pos[user_id]], index=item_ids)
    scores = scores.drop(labels=list(seen), errors="ignore")

    return scores.sort_values(ascending=False)

In [14]:
for user in sample_users:
    print("\nUser:", user)
    print("-" * 14)
    print("Top 5 Collaborative recommendations:")
    print("-" * 37)
    display(show_recommendations(cf_scores(user)))


User: user_03
--------------
Top 5 Collaborative recommendations:
-------------------------------------


,title,genres,score
0,Second Chances,drama|romance,2.555556
1,Weekend Alibi,comedy|mystery,2.555556
2,Night Archive,mystery|drama,2.555556
3,Wild Current,adventure|drama,2.555556
4,Skyline Chase,action|adventure,2.555556



User: user_10
--------------
Top 5 Collaborative recommendations:
-------------------------------------


,title,genres,score
0,The Last Signal,science fiction|thriller,2.555556
1,Cedar Street,drama|comedy,2.555556
2,Weekend Alibi,comedy|mystery,2.555556
3,Deep Blue Rescue,adventure|thriller,2.555556
4,Wild Current,adventure|drama,2.555556


# **Activity 4: Compare both recommenders**

In [15]:
# Compare the top-five lists
for user in sample_users:
    print("\nUser:", user)
    print("-" * 14)

    content_top = show_recommendations(content_scores(user, train)).assign(method="Content-Based")
    collaborative_top = show_recommendations(cf_scores(user)).assign(method="Collaborative")

    comparison = pd.concat([content_top, collaborative_top], ignore_index=True)

    display(comparison[["method", "title", "genres", "score"]])
    
    content_set = set(content_top["title"])
    collaborative_set = set(collaborative_top["title"])

    print("\nCommon recommendations:")
    print(content_set & collaborative_set)

    print("\nContent-based only:")
    print(content_set - collaborative_set)

    print("\nCollaborative only:")
    print(collaborative_set - content_set)


User: user_03
--------------


,method,title,genres,score
0,Content-Based,Orbiting Us,science fiction|romance,0.076052
1,Content-Based,Wild Current,adventure|drama,0.014665
2,Content-Based,Skyline Chase,action|adventure,0.009150
3,Content-Based,Second Chances,drama|romance,0.005498
4,Content-Based,Night Archive,mystery|drama,0.005463
5,Collaborative,Second Chances,drama|romance,2.555556
6,Collaborative,Weekend Alibi,comedy|mystery,2.555556
7,Collaborative,Night Archive,mystery|drama,2.555556
8,Collaborative,Wild Current,adventure|drama,2.555556
9,Collaborative,Skyline Chase,action|adventure,2.555556



Common recommendations:
{'Wild Current', 'Second Chances', 'Night Archive', 'Skyline Chase'}

Content-based only:
{'Orbiting Us'}

Collaborative only:
{'Weekend Alibi'}

User: user_10
--------------


,method,title,genres,score
0,Content-Based,Red Planet Run,science fiction|action,0.079308
1,Content-Based,Deep Blue Rescue,adventure|thriller,0.056889
2,Content-Based,The Last Signal,science fiction|thriller,0.032964
3,Content-Based,A Very Small Planet,science fiction|adventure,0.027449
4,Content-Based,Wild Current,adventure|drama,0.009994
5,Collaborative,The Last Signal,science fiction|thriller,2.555556
6,Collaborative,Cedar Street,drama|comedy,2.555556
7,Collaborative,Weekend Alibi,comedy|mystery,2.555556
8,Collaborative,Deep Blue Rescue,adventure|thriller,2.555556
9,Collaborative,Wild Current,adventure|drama,2.555556



Common recommendations:
{'Wild Current', 'The Last Signal', 'Deep Blue Rescue'}

Content-based only:
{'A Very Small Planet', 'Red Planet Run'}

Collaborative only:
{'Cedar Street', 'Weekend Alibi'}


**Comparison -->**
1. Content-based filtering uses movie descriptions, genres, and the user's own rating history.
2. Collaborative filtering uses patterns in the training ratings across users and movies.
3. The lists can differ because content similarity and predicted ratings measure different things.
4. Content-based filtering can recommend a new movie if its descriptive features are available.
5. Collaborative filtering can discover patterns between users and movies, but a new movie has no learned interaction pattern in this model.

# **Activity 5: Precision@5, Recall@5, and NDCG@5**

In [22]:
# Define the NDCG calculation
def dcg_at_k(relevances, k=5):
    rel = np.asarray(relevances[:k], dtype=float)

    if len(rel) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum((2 ** rel - 1) / discounts))

In [23]:
# Calculate metrics for both method
RELEVANCE_THRESHOLD = 3
K = 5

def evaluate_ranking(user_id, method, k=5):
    held = test[test["user_id"] == user_id]
    relevant = set(held.loc[held["rating"] >= RELEVANCE_THRESHOLD, "movie_id"])

    if method == "content":
        scores = content_scores(user_id, train)
    else:
        scores = cf_scores(user_id)

    recommended = list(scores.head(k).index)
    hits = [int(movie in relevant)
           for movie in recommended]
    precision = sum(hits) / k

    if len(relevant) > 0:
        recall = sum(hits) / len(relevant)
    else:
        recall = np.nan

    dcg = dcg_at_k(hits, k)

    ideal_relevances = [1] * min(len(relevant), k)

    ideal_dcg = dcg_at_k(ideal_relevances, k)

    if ideal_dcg > 0:
        ndcg = dcg / ideal_dcg
    else:
        ndcg = np.nan

    return precision, recall, ndcg, len(relevant)

In [24]:
metric_rows = []

for method in ["content", "collaborative"]:
    for user in user_ids:
        precision, recall, ndcg, relevant_count = (evaluate_ranking(user, method, K))

        metric_rows.append({
            "method": method, "user_id": user, "precision@5": precision,
            "recall@5": recall, "NDCG@5": ndcg, "relevant_test_items": relevant_count
        })

metrics = pd.DataFrame(metric_rows)

metrics

,method,user_id,precision@5,recall@5,NDCG@5,relevant_test_items
0,content,user_01,0.2,1.0,0.430677,1
1,content,user_02,0.4,1.0,0.650921,2
2,content,user_03,0.0,NaN,NaN,0
3,content,user_04,0.2,1.0,1.000000,1
4,content,user_05,0.4,1.0,0.624051,2
5,content,user_06,0.0,NaN,NaN,0
6,content,user_07,0.0,NaN,NaN,0
7,content,user_08,0.0,NaN,NaN,0
8,content,user_09,0.0,NaN,NaN,0
9,content,user_10,0.0,0.0,0.000000,1


In [25]:
# Summarise the result
summary = metrics.groupby("method").agg({
    "precision@5": "mean",
    "recall@5": "mean",
    "NDCG@5": "mean"
})

display(summary.round(4))

,precision@5,recall@5,NDCG@5
method,,,
collaborative,0.1167,0.6667,0.4054
content,0.1500,0.8333,0.5992


In [26]:
for method in ["content", "collaborative"]:
    method_metrics = metrics[metrics["method"] == method]
    valid = method_metrics.dropna(subset=["recall@5", "NDCG@5"])

    print("\nMethod:", method)
    print("Users evaluated for Precision@5:", len(method))
    print("Users contribute to recall@5, NDCG@5:", len(valid))

print("\nRelevance threshold: rating >= 3")
print("Held out split: 2 ratings per user")
print("Ranking cutoff: K = 5")


Method: content
Users evaluated for Precision@5: 7
Users contribute to recall@5, NDCG@5: 15

Method: collaborative
Users evaluated for Precision@5: 13
Users contribute to recall@5, NDCG@5: 15

Relevance threshold: rating >= 3
Held out split: 2 ratings per user
Ranking cutoff: K = 5


# **Activity 6: Cold start problem**

In [28]:
# Demonstrate recommendations for a new movie
new_movie = {
    "title": "Garden on Europa",
    "genres": "science fiction|drama",
    "description": "A botanist builds a greenhouse on an icy moon."
}

new_text = (new_movie["title"] + " " + 
           new_movie["genres"].replace("|", " ") + " " +
           new_movie["description"])

new_vector = vectorizer.transform([new_text])
new_similarity = cosine_similarity(new_vector, item_vectors).ravel()

top_indices = np.argsort(new_similarity)[::-1][:5]

new_movie_neighbours = movies.iloc[top_indices][["title", "genres"]].copy()
new_movie_neighbours["similarity"] = new_similarity[top_indices]

display(new_movie_neighbours)

,title,genres,similarity
0,Moon Harbor,science fiction|drama,0.512326
14,A Very Small Planet,science fiction|adventure,0.228048
11,Orbiting Us,science fiction|romance,0.141576
2,The Last Signal,science fiction|thriller,0.133596
1,Red Planet Run,science fiction|action,0.121801


**Cold Start Solution -->**
1. `New user:-` Ask the user to select favourite movies or genres. Recommend popular movies until enough ratings are collected.
2. `New Movie:-` Use the title, description, and genre tags to recommend the movie through content-based filtering.
3. `Collaborative Filtering:-` Collect user ratings and interactions so the model can learn patterns for new users and movies.
4. `Fallback:-` Show popular or highly rated movies when there is insufficient information for personalization.

# **Activity 7: Implicit feedback vs explicit ratings**

**Explicit Feedback :-**
* Users directly express their preferences through rating, such as 1 to 5 stars.

**Implicit Feedback :-**
* User behaviour provides indirect signals, such as clicks, watch time, purchases, and searches.

**Important Difference :-**
* An explicit low rating can indicate dislike. An unobserved movie is not necessarily disliked because the user may never have seen it.
* Implicit feedback can also be ambiguous. A click does not always mean the user liked the movie. Exposure, accidental clicks, and watch duration can affect interpretation.

**Conclusion -->**
* In this task, I implemented two recommendation systems: content-based filtering using TF-IDF and collaborative filtering using TruncatedSVD.
* Content-based filtering uses movie descriptions, genres, and a user's rating history to recommend similar movies.
* Collaborative filtering uses patterns in the user-item ratings matrix to predict ratings for unseen movies.
* RMSE evaluates rating prediction accuracy, while Precision@5, Recall@5, and NDCG@5 evaluate recommendation ranking quality.
* The two methods can produce different recommendations because they rely on different evidence.
* Cold start can be handled through onboarding questions, movie metadata, and popular-item fallback recommendations.
* Implicit feedback provides behavioural information, but an unobserved movie should not automatically be treated as a negative example.
* The dataset used here is synthetic and small, so the results demonstrate the methods rather than guaranteeing real-world recommendation quality.